# **Implement the data processing, data visualization and data wrangling on any real world problem i.e., Covid_19 dataset to view the active cases on the world map using the Choropleth and also plot the cases**

### **Visualization of a Choropleth Map to Show the Details of COVID-19 Cases for Indian States and Union Territories. It displays total cases, active cases, and total deaths in each Indian state. Try to render a map that is as interactive as possible.**

1. Geospatial data visualization and analysis using Folium library.
2. COVID-19 data is accessed through an API.
3. Coordinates for Indian states are traced using GeoJSON.

(https://python-visualization.github.io/folium/),(https://data.covid19india.org/),(https://geojson.io/)

### **A Brief Introduction to Folium**

folium builds on the data wrangling strengths of the Python ecosystem and the mapping strengths of the leaflet js library. Maripulate your data in Python, then visualize it in on a Leaflet map via folium. Leaflet is the leading open-source JavaScript library for mobile-friendly interactive maps.

Folium makes it easy to visualize data that's been manipulated in Python on an interactive leaflet map. It enables both the binding of data to a map for choropleth visualizations as well as passing rich vector/raster/HTML visualizations as markers on the map. A choropleth map is a thematic map in which areas are shaded or patterned in proportion to the measurement of the statistical variable being displayed on the map, such as population density or per-capita income.

The library has a number of built-in tilesets from OpenStreetMap, Mapbox, and Stamen, and supports custom tilesets with Mapbox or Cloudmade API keys. folium supports both Image, Video, GeoJSON and TopoJSON overlays.

### **Importing required Modules/Libraries**

In [ ]:
pip install folium

#### **import json - This imports the built-in "json" module for working with JSON data**

#### **import requests - This Imports the "requests" library for making HTTP requests and interacting with APIs**

The plugins module in Follum provides additional functionality that can be used to enhance the interactivity and visualization of maps. This module contains a set of plugins that can be added to a Folum map to extend its capabilities

For example, the MarkerCluster plugin can be used to group markers that are close together on the map into a single cluster, making it easier to visualize large numbers of markers. The HeatMap plugin can be used to create heat maps of geospatial data, which can be used to identify hotspots or areas of high concentration.

Other plugins in the plugins module include Fullscreen, Draw, MeasureControl, MiniMap, and many more. Each plugin has its own set of features and updons that can be customized to meet specific mapping needs.

In [2]:
import folium as fle
from folium import plugins
import pandas as pd
import json
import requests

## **Loading Geo-JSON data**

This data consists of co-ordinates which is used to plot the map.

In [3]:
with open(r'Indian_States.json') as f:
    geojson_states = json.load(f)

## **Assigning name of states to a key called 'id'**

In [4]:
for i in geojson_states['features']:
    i['id'] = i['properties']['NAME_1']

## **Step 1: load COVID - 19 Data from Local JSON File**

## **The API is no longer available, so we are loading the proviously downloaded.**

## **COVID-19 dataset from a local JSON file.**

## **This ensures data availability even if the online source is down.**

In [5]:
with open("data.json", "r") as file:
    covid_current = json.load(file)

## **Step 2: Convert JSON Data to Pandas DataFrame**

## **Instead of using the deprecated .append() method, we create a lost of dictionaries to store state-wise COVID-19 Data.**

## **The first entry in the JSON (total cases for the entire country) is skipped.**

## **We only keep data for individual states, excluding "State Unassigned"**

## **Finally, the list is converted into a Pandas DataFrame for further processing.**

In [6]:
import pandas as pd

# Create a list of dictionaries instead of using append()
data_list = []
for state_data in covid_current["statewise"][1:]: #Skipping the first entry (total Cases)
    if state_data["state"] != "State Unassigned":
        data_list.append({
            "State": state_data["state"],
            "Total Cases": int(state_data["confirmed"]),
            "Active Cases": int(state_data["active"]),
            "Deaths": int(state_data["deaths"])
        })

#convert the List into a DataFrame
df_covid = pd.DataFrame(data_list)

In [7]:
df_covid.rename(columns={
    "Total Cases": "total_case",
    "Active Cases": "active_case",
    "Deaths": "total_deaths"
}, inplace=True)

## **Step 3: Verify and Correct State Name Mismatches**

## **The COVID-19 Dataset (df_covid) and the GeoJSON File (NAME_1 property).**

This step ensures that all state names in the dataset match those in the GeoJSON file:

1. Extracts the list of state names from the GeoJSON file (NAME_1 property).

2. Extracts the list of state names from the COVID-19 DataFrame.

3. Identifies states that are present in GeoJSON but missing in df_covid.

4. Manually corrects known mismatches (e.g., "Dadra and Nagar Haveli and Daman and Diu" is renamed to "Dadra and Nagar Haveli" to ensure consistency).

This correction ensures that every state is correctly mapped to its respective COVID-19 data when visualized on the choropleth map.



In [13]:
# Get List of states from GeoJSON
geojson_states_names = [feature["properties"]["NAME_1"] for feature in geojson_states["features"]]

# Get list of states from DataFrame 
df_covid_states = df_covid["State"].unique()

# Find missing states
missing_states = set(geojson_states_names) - set(df_covid_states) 
print("States present in Geo350N but missing in Dataframe:", missing_states)

#Fix known mismatches manually (Example)
state_name_corrections = {
    "Dadra and Nagar Haveli and Daman and Diu": "Dadra and Nagar Haveli",
    "NCT of Delhi": "Delhi"
}

df_covid["State"] = df_covid["State"].replace(state_name_corrections)

States present in Geo350N but missing in Dataframe: set()


## **Step 4: Map COVID-19 Data to GeoJSON Features**

## **This step ensures that each state in the GeoJSOnfike correctly assigned its corresponding COVID-19**

## **The first entry in the JSON (total cases for the entire country) is skipped.**

## **We only keep data for individual states, excluding "State Unassigned"**

## **Finally, the list is converted into a Pandas DataFrame for further processing.**

In [15]:
# create a lookup dictionary from DataFrame
state_data_map = df_covid.set_index("State").to_dict(orient="index")

# Assign values to GeoJSON features dynamically
for feature in geojson_states['features']:
    state_name = feature['properties']['NAME_1']
    if state_name in state_data_map:
        feature['properties']['total_case']= state_data_map[state_name]['total_case']
        feature['properties']['active_case'] = state_data_map[state_name]['active_case']
        feature['properties']['total_deaths'] = state_data_map[state_name]['total_deaths']
    else:
        feature['properties']['total_case'] = "No Data"
        feature['properties']['active_case'] = "No Data"
        feature['properties']['total_deaths'] = "No Data"

In [ ]:
# STEP 5: Initialize the Folium Map
"""Creates an iteractive map centered on India using Mapbox tiles for better visualization.
A marker is added at India's Geographic centre for reference."""

In [ ]:
# Replace "YOUR_MAPBOX_TOKEN" with your actual MApbox access token
map1 = flm.Map(location=[20.5937,78.9629], zoom_start=4, tiles= "https://api.mapbox.com/styles/v1/mapbox/outdoors-v11/tiles/{z}/{x}/{y}?access_token=pk.eyJ1IjoibGFrc2hpeXRhLTAwNCIsImEiOiJjbTd5YXVsZ3IwOWZyMmpzNXd3Nm4xeXhnIn0.B9zVVBQLHGvralir72HIgg", attr="Mapbox Bright")
#Add a  marker to the map
flm.Marker([20.5937,78.9629], popup='India').add_to(map1)

#Display the map
map1

In [ ]:
step 6 add multiple tile laers

adds different map tile styles (eg: watercolor terrain and open street map) to engance visualisation

In [ ]:
choropleth = flm.Choropleth(
    geo_data=geojson_states,
    name="Total Cases",
    data=df_covid,
    columns=["State", "total_case"],
    key_on="feature.properties.NAME_1",
    fill_color="Y1OrRd",
    fill_opacity=0.7,
    nan_fill_color="white",
    legend_name="State-wise COVID-19 Cases in India"
).add_to(map1)
    

In [ ]:
map1

In [ ]:
choropleth.geojson.add_child(
    flm.features.GeoJsonTooltip(
        fields=["NAME_1", "total_case", "active_case", "total_deaths"],
        aliases=["States", "Total Cases", "Active Cases", "Deaths"],
        labels=True,
        sticky=True
    )
)

In [ ]:
map1.save("IndianMap.html")
map1